# Visão Computacional com CNNs e Transformers
## 🎓 Faculdade Infnet — Pós-Graduação
### 🎯 Laboratório Prático: Detecção de Objetos One-Stage com YOLO (Inferência Pré-Treinada, Fine-Tuning e Análise de Saídas)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_01_cnn_architectures/aula_01_yolo_detection.ipynb)

---

### 📖 Contextualização Pedagógica: O Paradigma One-Stage

Enquanto detectores *Two-Stage* (como o **Faster R-CNN**) utilizam uma etapa separada para propor regiões de interesse (RPN) antes de classificá-las, a família **YOLO (You Only Look Once)** revolucionou a visão computacional ao formular a detecção de objetos como um **problema único de regressão direta**.

A rede processa a imagem inteira em uma **única passada convolucional (*One-Stage*)**, dividindo o espaço em uma grade e prevendo simultaneamente:
1. As coordenadas contínuas das caixas delimitadoras $[x_c, y_c, w, h]$.
2. A confiança de que há um objeto na região (*Objectness*).
3. A distribuição de probabilidades sobre as classes.

Essa arquitetura permite atingir velocidades extremas de **30 a 120+ FPS** em tempo real, tornando o YOLO o padrão mais popular da indústria para veículos autônomos, robótica móvel, monitoramento de segurança e dispositivos de borda (*Edge AI*).

---

### 🎯 Objetivos de Aprendizagem
1. **Inferência Direta (*Off-the-Shelf / Zero-Shot*)**: Aplicar o modelo YOLO pré-treinado no dataset MS-COCO (80 classes) diretamente sobre imagens reais pré-existentes, sem necessidade de treino.
2. **Desmistificar a Estrutura de Saída do YOLO**: Inspecionar os tensores de predição `boxes.xyxy` (pixels absolutos), `boxes.xywhn` (normalizados), `boxes.conf` (confianças) e `boxes.cls` (IDs de classe).
3. **Pipeline de Dados no Formato YOLO**: Baixar um dataset do Kaggle via `kagglehub` e construir a estrutura de diretórios e o manifesto `dataset.yaml` no formato padrão YOLO TXT (`class_id x_center y_center width height`).
4. **Fine-Tuning e Transfer Learning**: Treinar o modelo no domínio customizado, monitorando métricas essenciais (**mAP@50**, **mAP@50-95**, Box Loss, Cls Loss e DFL Loss).
5. **Inferência Visual Pós-Treino**: Carregar o melhor checkpoint (`best.pt`) e renderizar caixas delimitadoras estilizadas sobre imagens de teste.
6. **Comparativo Arquitetural**: Analisar os trade-offs de engenharia entre **One-Stage (YOLO)** e **Two-Stage (Faster R-CNN)**.


## 1. Configuração do Ambiente e Instalação de Dependências

Vamos instalar a biblioteca oficial **`ultralytics`** e o **`kagglehub`**, além de verificar o acelerador de GPU CUDA no Google Colab.


In [ ]:
# Instalação das bibliotecas necessárias no Google Colab
!pip install -q ultralytics kagglehub

import os
import sys
import time
import shutil
import urllib.request
import xml.etree.ElementTree as ET
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
from ultralytics import YOLO

# Fixar sementes para reprodutibilidade
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Verificação de Dispositivo
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🔥 Dispositivo de Execução: {device.upper()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("   ⚠️ Atenção: Nenhuma GPU detectada. No Colab, vá em: Ambiente de Execução -> Alterar tipo de ambiente -> T4 GPU")


## 2. Parte 1 — Inferência Direta com YOLO Pré-Treinado no MS-COCO (Zero-Shot)

Antes de qualquer treinamento, podemos carregar os pesos oficiais do **YOLOv8 Nano (`yolov8n.pt`)** pré-treinado nas 80 classes do dataset **MS-COCO** (pessoas, carros, animais, bicicletas, frutas, móveis, etc.).

O modelo Nano possui apenas **3.2 milhões de parâmetros**, sendo extremamente leve e rápido (~2 a 5 ms por imagem em GPU).


In [ ]:
# 1. Carregar o modelo YOLOv8 Nano pré-treinado no MS-COCO
pretrained_model = YOLO('yolov8n.pt')

print("✅ Modelo YOLOv8n carregado com sucesso!")
print(f"   • Total de Classes Pré-Treinadas (COCO): {len(pretrained_model.names)}")
print(f"   • Amostra de Classes: {[pretrained_model.names[i] for i in range(10)]}")


### 2.1 Baixando Imagens de Teste do Mundo Real

Vamos baixar imagens públicas contendo cenas variadas (uma rua movimentada com pessoas e carros, e um animal de estimação) para testar o detector pré-treinado diretamente.


In [ ]:
# Criar pasta para imagens de teste
os.makedirs("sample_images", exist_ok=True)

# URLs de imagens públicas de teste (alta resolução do Wikimedia / Pexels)
IMAGE_URLS = {
    "traffic.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d3/City_traffic_in_New_York_City.jpg/800px-City_traffic_in_New_York_City.jpg",
    "cat_dog.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Cat_August_2010-4.jpg/800px-Cat_August_2010-4.jpg"
}

for img_name, url in IMAGE_URLS.items():
    dest_path = os.path.join("sample_images", img_name)
    if not os.path.exists(dest_path):
        urllib.request.urlretrieve(url, dest_path)
        print(f"📥 Imagem salva em: {dest_path}")
    else:
        print(f"✅ Imagem pronta em: {dest_path}")


### 2.2 Executando a Inferência Direta e Inspecionando as Saídas

Vamos rodar a inferência sobre a imagem de tráfego urbano e desmistificar a estrutura de objetos retornada pelo YOLO.


In [ ]:
# Executar inferência sobre a imagem de tráfego
traffic_path = os.path.join("sample_images", "traffic.jpg")
results = pretrained_model.predict(source=traffic_path, conf=0.35, device=device)

# O objeto 'results' é uma lista contendo os resultados para cada imagem processada
res = results[0]

print("🔍 Estrutura do Objeto de Resultados do YOLO:")
print(f"   • Formato da Imagem Original: {res.orig_shape} (Altura, Largura)")
print(f"   • Total de Caixas Detectadas:  {len(res.boxes)}")
print(f"   • Tempo de Inferência (ms):    {res.speed['inference']:.2f} ms")


### 2.3 Desmistificando os Tensores de Saída do YOLO

Cada detecção no YOLO é composta por 4 tensores fundamentais acessíveis via `res.boxes`:

1. **`boxes.xyxy`** (`Tensor[N, 4]`): Coordenadas das caixas em **pixels absolutos** $[x_{\text{min}}, y_{\text{min}}, x_{\text{max}}, y_{\text{max}}]$.
2. **`boxes.xywhn`** (`Tensor[N, 4]`): Coordenadas **normalizadas** $[x_c, y_c, w, h] \in [0.0, 1.0]$ relativas à largura e altura da imagem.
3. **`boxes.conf`** (`Tensor[N]`): Pontuação de confiança da detecção (probabilidade Softmax $\in [0.0, 1.0]$).
4. **`boxes.cls`** (`Tensor[N]`): ID numérico inteiro da classe detectada.
5. **`names`** (`dict`): Dicionário mapeando cada ID numérico ao nome textual da classe (ex: `0: 'person'`, `2: 'car'`).


In [ ]:
# Extrair tensores de predição
boxes_xyxy = res.boxes.xyxy.cpu().numpy()
confidences = res.boxes.conf.cpu().numpy()
class_ids = res.boxes.cls.cpu().numpy().astype(int)
class_names = res.names

print("📋 Detalhes das 5 Primeiras Caixas Detectadas:")
print("─" * 80)
print(f"{'#':<3} | {'Classe':<15} | {'Confiança':<10} | {'Coordenadas [x1, y1, x2, y2]'}")
print("─" * 80)

for i in range(min(5, len(boxes_xyxy))):
    cls_name = class_names[class_ids[i]]
    conf = confidences[i]
    box = boxes_xyxy[i]
    coords_str = f"[{box[0]:.1f}, {box[1]:.1f}, {box[2]:.1f}, {box[3]:.1f}]"
    print(f"{i+1:<3} | {cls_name:<15} | {conf*100:.1f}%{'':<5} | {coords_str}")
print("─" * 80)


### 2.4 Renderização Visual da Inferência Pré-Treinada

Podemos visualizar as predições utilizando a renderização automática do Ultralytics (`res.plot()`) ou desenhando com Matplotlib.


In [ ]:
# 1. Renderizar com o motor visual nativo do Ultralytics
annotated_img_bgr = res.plot() # Retorna matriz NumPy em formato BGR
annotated_img_rgb = annotated_img_bgr[:, :, ::-1] # Converter BGR para RGB

plt.figure(figsize=(12, 7))
plt.imshow(annotated_img_rgb)
plt.title(f"Inferência Direta com YOLOv8n (COCO Pré-Treinado) • {len(boxes_xyxy)} objetos detectados", fontsize=13, fontweight='bold', color='#0A345D')
plt.axis('off')
plt.tight_layout()
plt.show()

# 2. Testar na imagem do gato
cat_path = os.path.join("sample_images", "cat_dog.jpg")
cat_res = pretrained_model.predict(source=cat_path, conf=0.5, device=device)[0]

plt.figure(figsize=(9, 6))
plt.imshow(cat_res.plot()[:, :, ::-1])
plt.title("Detecção Pré-Treinada: Animal Doméstico", fontsize=12, fontweight='bold', color='#0A345D')
plt.axis('off')
plt.tight_layout()
plt.show()


## 3. Parte 2 — Download do Dataset de Fine-Tuning via `kagglehub`

Agora que dominamos o uso do modelo pré-treinado, vamos realizar o **Fine-Tuning** do YOLO em um dataset customizado com 3 classes de frutas do mundo real (**Maçãs, Bananas e Laranjas**).

Baixamos o dataset **`mbkinaci/fruit-images-for-object-detection`** diretamente com a biblioteca oficial **`kagglehub`**.


In [ ]:
import kagglehub

print("📥 Baixando dataset do Kaggle via kagglehub...")
start_time = time.time()

dataset_path = kagglehub.dataset_download("mbkinaci/fruit-images-for-object-detection")

print(f"✅ Download concluído em {time.time() - start_time:.2f}s!")
print(f"📂 Diretório Local do Dataset: {dataset_path}")


## 4. Conversão das Anotações para o Formato Padrão YOLO TXT

O dataset original possui anotações em formato Pascal VOC XML (`[xmin, ymin, xmax, ymax]`). O YOLO exige que cada imagem `.jpg` possua um arquivo `.txt` homônimo, onde cada linha representa um objeto no formato normalizado:

$$\text{linha} = [\text{class\_id}, \; x_c, \; y_c, \; w, \; h]$$

Onde:
- $x_c = \frac{x_{\text{min}} + x_{\text{max}}}{2 \times \text{largura}}$ (Centro Horizontal normalizado $\in [0, 1]$)
- $y_c = \frac{y_{\text{min}} + y_{\text{max}}}{2 \times \text{altura}}$ (Centro Vertical normalizado $\in [0, 1]$)
- $w = \frac{x_{\text{max}} - x_{\text{min}}}{\text{largura}}$ (Largura normalizada $\in [0, 1]$)
- $h = \frac{y_{\text{max}} - y_{\text{min}}}{\text{altura}}$ (Altura normalizada $\in [0, 1]$)

Vamos criar a estrutura oficial do YOLO em disco:


In [ ]:
# Mapeamento de Classes para o YOLO (Inicia em 0)
YOLO_CLASSES = {
    'apple': 0,
    'banana': 1,
    'orange': 2
}

# Criar estrutura de pastas padrão do YOLO
yolo_dataset_dir = os.path.abspath("yolo_fruit_dataset")
for split in ['train', 'val']:
    os.makedirs(os.path.join(yolo_dataset_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(yolo_dataset_dir, 'labels', split), exist_ok=True)

def convert_voc_to_yolo(source_folder, split_name):
    """
    Lê imagens e XMLs Pascal VOC e converte para a pasta padrão do YOLO com arquivos .txt normalizados.
    """
    count = 0
    for filename in os.listdir(source_folder):
        if filename.endswith(('.jpg', '.png', '.jpeg')):
            base_name = os.path.splitext(filename)[0]
            xml_path = os.path.join(source_folder, base_name + ".xml")
            img_path = os.path.join(source_folder, filename)
            
            if not os.path.exists(xml_path):
                continue
                
            img = Image.open(img_path)
            img_w, img_h = img.size
            
            # Copiar imagem para a pasta do YOLO
            dest_img_path = os.path.join(yolo_dataset_dir, 'images', split_name, filename)
            shutil.copyfile(img_path, dest_img_path)
            
            # Converter XML para TXT
            tree = ET.parse(xml_path)
            root = tree.getroot()
            yolo_lines = []
            
            for obj in root.findall('object'):
                cls_name = obj.find('name').text.lower().strip()
                if cls_name in YOLO_CLASSES:
                    cls_id = YOLO_CLASSES[cls_name]
                    bndbox = obj.find('bndbox')
                    xmin = float(bndbox.find('xmin').text)
                    ymin = float(bndbox.find('ymin').text)
                    xmax = float(bndbox.find('xmax').text)
                    ymax = float(bndbox.find('ymax').text)
                    
                    # Calcular coordenadas normalizadas [xc, yc, w, h]
                    xc = ((xmin + xmax) / 2.0) / img_w
                    yc = ((ymin + ymax) / 2.0) / img_h
                    w = (xmax - xmin) / img_w
                    h = (ymax - ymin) / img_h
                    
                    # Garantir limites [0, 1]
                    xc, yc = max(0.0, min(1.0, xc)), max(0.0, min(1.0, yc))
                    w, h = max(0.0, min(1.0, w)), max(0.0, min(1.0, h))
                    
                    yolo_lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
                    
            # Salvar arquivo .txt com os rótulos do YOLO
            dest_txt_path = os.path.join(yolo_dataset_dir, 'labels', split_name, base_name + ".txt")
            with open(dest_txt_path, 'w') as f:
                f.write("\n".join(yolo_lines))
            count += 1
            
    return count

# Localizar pastas descompactadas do Kaggle
raw_train_dir = os.path.join(dataset_path, "train_zip", "train") if os.path.exists(os.path.join(dataset_path, "train_zip", "train")) else os.path.join(dataset_path, "train")
raw_test_dir = os.path.join(dataset_path, "test_zip", "test") if os.path.exists(os.path.join(dataset_path, "test_zip", "test")) else os.path.join(dataset_path, "test")

n_train = convert_voc_to_yolo(raw_train_dir, 'train')
n_val = convert_voc_to_yolo(raw_test_dir, 'val')

print(f"✅ Conversão Concluída com Sucesso:")
print(f"   • Imagens de Treino (YOLO): {n_train}")
print(f"   • Imagens de Validação:     {n_val}")


## 5. Criação do Arquivo de Configuração `dataset.yaml`

O YOLO requer um arquivo YAML que especifica os caminhos dos diretórios de treino e validação, o número de classes (`nc`) e a lista com os nomes das classes (`names`).


In [ ]:
# Criar arquivo data.yaml
yaml_content = f"""path: {yolo_dataset_dir}
train: images/train
val: images/val

# Classes
names:
  0: Maçã (Apple)
  1: Banana
  2: Laranja (Orange)
"""

yaml_file_path = os.path.join(yolo_dataset_dir, "fruit_data.yaml")
with open(yaml_file_path, "w") as f:
    f.write(yaml_content)

print(f"📄 Arquivo YAML gerado em: {yaml_file_path}")
print("─" * 40)
print(yaml_content)
print("─" * 40)


## 6. Fine-Tuning do YOLOv8 no Dataset Customizado de Frutas

Iniciamos o treinamento com o método `model.train()`. 

### ⚙️ Hiperparâmetros:
- `data`: Caminho do arquivo YAML.
- `epochs`: 10 épocas (suficiente para convergência com pesos pré-treinados).
- `imgsz`: 640x640 (resolução padrão do YOLO).
- `batch`: 16 imagens por batch.
- `device`: GPU CUDA (`0` ou `cuda`).


In [ ]:
# Instanciar um modelo novo para o fine-tuning
ft_model = YOLO('yolov8n.pt')

print("🚀 Iniciando Fine-Tuning do YOLOv8n...")
start_train = time.time()

# Executar treinamento
train_results = ft_model.train(
    data=yaml_file_path,
    epochs=10,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    plots=True,
    verbose=True
)

print(f"\n🎉 Fine-Tuning concluído em {time.time() - start_train:.2f} segundos!")
print(f"📂 Checkpoints e Métricas salvos em: {train_results.save_dir}")


## 7. Análise das Métricas de Detecção (mAP@50, mAP@50-95 e Losses)

Durante o treino, o YOLO gera automaticamente o gráfico `results.png` com a evolução das perdas de caixa (*box_loss*), classe (*cls_loss*), *dfl_loss* e a métrica de precisão média **mAP@50** e **mAP@50-95**.


In [ ]:
# Carregar e exibir o gráfico de resultados gerado pelo Ultralytics
results_plot_path = os.path.join(train_results.save_dir, "results.png")

if os.path.exists(results_plot_path):
    plt.figure(figsize=(14, 8))
    results_img = Image.open(results_plot_path)
    plt.imshow(results_img)
    plt.title("Evolução das Métricas e Perdas do YOLO (Fine-Tuning)", fontsize=13, fontweight='bold', color='#0A345D')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Gráfico de resultados salvo em:", train_results.save_dir)


## 8. Inferência Visual com o Modelo Customizado (`best.pt`)

Carregamos o melhor checkpoint salvo (`best.pt`) e executamos predições sobre imagens de validação do conjunto de frutas.


In [ ]:
# Carregar o melhor modelo salvo pelo fine-tuning
best_weight_path = os.path.join(train_results.save_dir, "weights", "best.pt")
custom_yolo = YOLO(best_weight_path)

# Testar inferência em 3 imagens da pasta de validação
val_images_dir = os.path.join(yolo_dataset_dir, 'images', 'val')
sample_val_imgs = [os.path.join(val_images_dir, f) for f in os.listdir(val_images_dir)[:3]]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, img_path in enumerate(sample_val_imgs):
    # Executar predição com threshold de confiança 0.60
    preds = custom_yolo.predict(source=img_path, conf=0.60, device=device)[0]
    
    annotated = preds.plot()[:, :, ::-1] # BGR para RGB
    
    axes[idx].imshow(annotated)
    axes[idx].set_title(f"Amostra #{idx+1} • {len(preds.boxes)} fruta(s)", fontsize=11, fontweight='bold', color='#0A345D')
    axes[idx].axis('off')

plt.suptitle("🎯 Resultados de Inferência com YOLO Fine-Tuned (best.pt)", fontsize=13, fontweight='bold', color='#0A345D', y=0.98)
plt.tight_layout()
plt.show()


## 9. Comparativo Arquitetural: YOLO (One-Stage) vs Faster R-CNN (Two-Stage)

A tabela a seguir sintetiza os compromissos de engenharia (*trade-offs*) entre as duas grandes famílias de detectores:

| Critério de Engenharia | One-Stage (Família YOLO) | Two-Stage (Faster R-CNN) |
| :--- | :--- | :--- |
| **Paradigma** | Passada única direta em grade convolucional | Proposta de regiões (RPN) + Classificação RoI |
| **Velocidade de Inferência** | **Ultra-Rápido (30 a 120+ FPS)** | Moderado (5 a 18 FPS) |
| **Precisão em Objetos Minúsculos** | Boa (otimizada em versões recentes com P2) | **Cirúrgica e Superior (RoIAlign preciso)** |
| **Consumo de Hardware** | Leve (modelos Nano/Small rodam em celulares e Raspberry Pi) | Mais pesado (exige GPU robusta para treinamento e inferência) |
| **Complexidade de Pipeline** | Simples (end-to-end com saída direta) | Mais complexo (Loss multitarefa em 2 estágios com RoIPooling) |
| **Aplicações Ideais** | Carros autônomos, robôs móveis, monitoramento por drones e vídeo em tempo real | Diagnóstico por imagens médicas (raio-x, tomografias), satélites e perícia visual |


## 10. 🎓 Desafios Práticos & Exercícios Propostos (Pós-Graduação)

Para consolidar o aprendizado prático:

---

### 🏋️‍♂️ Desafio 1: Variação de Portes de Modelo (Nano vs Small vs Medium)
Substitua o backbone `yolov8n.pt` por `yolov8s.pt` (Small) e `yolov8m.pt` (Medium). Compare o tempo de treino por época e a variação da métrica **mAP@50-95**.

### 🏋️‍♂️ Desafio 2: Inferência em Vídeo em Tempo Real
Utilize a API do Ultralytics para processar um arquivo de vídeo `.mp4` com rastreamento integrado (`model.track(source='video.mp4', show=True)`). Observe como os IDs persistentes são atribuídos a cada fruta em movimento.

### 🏋️‍♂️ Desafio 3: Exportação para Produção (ONNX / TensorRT / OpenVINO)
Exporte o modelo treinado para o formato padrão de interoperabilidade **ONNX**:
```python
custom_yolo.export(format='onnx')
```
Meça a latência de inferência (em milissegundos) comparando o modelo PyTorch puro versus o runtime ONNX.
